# 🔴 Day 14 — Long-Term Memory (Store) & Runtime Context

**LangGraph from Zero to Advanced · Day 14 of 16**

**Today's goal:** remember facts about a user **across different conversations** (threads), and pass per-run settings like `user_id` safely.

| | Short-term memory (Day 7) | Long-term memory (today) |
|---|---|---|
| Tool | Checkpointer | **Store** |
| Scope | one thread (one chat) | across **all** threads |
| Holds | the message history | facts, preferences, profiles |
| Example | "what did I just say?" | "she's vegetarian and likes Python" |

| Lesson | Topic |
|---|---|
| 1 | Store basics: put / get / search |
| 2 | Runtime context: `user_id` per run |
| 3 | Use the store inside a graph |
| 4 | A chatbot that remembers you in every new chat |
| 5 | Semantic search over memories |
| 6 | Practice |

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — Store basics

A **Store** is a simple key-value database for JSON documents, organised by **namespace** (a tuple, like a folder path).

```
("users", "u42", "memories")  ──► key "food"  → {"fact": "vegetarian"}
                              └─► key "lang"  → {"fact": "loves Python"}
```

No LLM needed for this lesson.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
ns = ("users", "u42", "memories")                    # namespace = "folder"

store.put(ns, "food", {"fact": "is vegetarian"})     # put(namespace, key, value)
store.put(ns, "lang", {"fact": "loves Python"})

item = store.get(ns, "food")
print("get   :", item.key, item.value)

print("search:", [i.value["fact"] for i in store.search(ns)])   # list everything in a namespace

store.put(ns, "food", {"fact": "is vegan now"})      # same key = update
print("after update:", store.get(ns, "food").value)

store.delete(ns, "lang")
print("after delete:", [i.key for i in store.search(ns)])

## Lesson 2 — Runtime context: `user_id` per run

Things like `user_id`, `language` or a DB connection are **not** part of the conversation state — they're settings for **this run**. Declare a **context schema** and pass `context=` when invoking. Nodes read it through `runtime.context`.

In [ ]:
from dataclasses import dataclass
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime

@dataclass
class Context:
    user_id: str
    language: str = "English"

class GreetState(TypedDict):
    greeting: str

def greet(state: GreetState, runtime: Runtime[Context]) -> dict:   # 👈 extra `runtime` argument
    ctx = runtime.context
    return {"greeting": f"[{ctx.language}] Hello user {ctx.user_id}!"}

g = StateGraph(GreetState, context_schema=Context)
g.add_node("greet", greet)
g.add_edge(START, "greet")
g.add_edge("greet", END)
greet_app = g.compile()

print(greet_app.invoke({}, context=Context(user_id="u42")))
print(greet_app.invoke({}, context=Context(user_id="u7", language="Marathi")))

## Lesson 3 — Use the store inside a graph

Compile with `store=store`. Any node can then use `runtime.store`. Combine with `runtime.context.user_id` to read **that user's** memories.

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import SystemMessage

def personal_bot(state: MessagesState, runtime: Runtime[Context]) -> dict:
    ns = ("users", runtime.context.user_id, "memories")
    facts = [i.value["fact"] for i in runtime.store.search(ns)]
    system = "You are a helpful assistant. Personalise answers using what you know about the user: " + (", ".join(facts) or "nothing yet")
    return {"messages": [llm.invoke([SystemMessage(content=system)] + state["messages"])]}

g = StateGraph(MessagesState, context_schema=Context)
g.add_node("bot", personal_bot)
g.add_edge(START, "bot")
g.add_edge("bot", END)
bot_app = g.compile(store=store)

out = bot_app.invoke({"messages": [("user", "Suggest a dinner for tonight.")]}, context=Context(user_id="u42"))
print("🤖 (u42 — vegan):", out["messages"][-1].content)

out = bot_app.invoke({"messages": [("user", "Suggest a dinner for tonight.")]}, context=Context(user_id="stranger"))
print("\n🤖 (stranger):", out["messages"][-1].content)

## Lesson 4 — A chatbot that remembers you in every new chat

Now the bot **writes** memories too:

```
START ► chat (reads memories) ► remember (LLM extracts new facts → store) ► END
```

* Checkpointer = remembers **this** chat (Day 7)
* Store = remembers **you**, across chats

In [ ]:
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import InMemorySaver
import uuid

class NewFacts(BaseModel):
    facts: list[str] = Field(description="new long-term facts about the USER (name, job, likes, plans). Empty if none.")

extractor = llm.with_structured_output(NewFacts)
memory_store = InMemoryStore()

def chat(state: MessagesState, runtime: Runtime[Context]) -> dict:
    ns = ("users", runtime.context.user_id, "memories")
    facts = [i.value["fact"] for i in runtime.store.search(ns)]
    system = f"You are a friendly assistant. Known facts about the user: {facts or 'none'}. Keep replies short."
    return {"messages": [llm.invoke([SystemMessage(content=system)] + state["messages"])]}

def remember(state: MessagesState, runtime: Runtime[Context]) -> dict:
    last_user_msg = [m for m in state["messages"] if m.type == "human"][-1].content
    new = extractor.invoke(f"Extract long-term facts about the user from this message: {last_user_msg}")
    ns = ("users", runtime.context.user_id, "memories")
    for fact in new.facts:
        runtime.store.put(ns, str(uuid.uuid4()), {"fact": fact})
        print("💾 saved:", fact)
    return {}

g = StateGraph(MessagesState, context_schema=Context)
g.add_sequence([chat, remember])
g.add_edge(START, "chat")
g.add_edge("remember", END)
memory_bot = g.compile(checkpointer=InMemorySaver(), store=memory_store)
show_graph(memory_bot)

In [ ]:
me = Context(user_id="learner-1")

# Chat #1 (thread A)
memory_bot.invoke({"messages": [("user", "Hi! I'm Neha, a data engineer from Pune, and I love masala chai.")]},
                  {"configurable": {"thread_id": "thread-A"}}, context=me)

# Chat #2 — a BRAND NEW thread: no message history at all
out = memory_bot.invoke({"messages": [("user", "What do you know about me? Suggest a drink.")]},
                        {"configurable": {"thread_id": "thread-B"}}, context=me)
print("\n🤖 (new thread):", out["messages"][-1].content)

In [ ]:
print("All memories for 'learner-1':")
for item in memory_store.search(("users", "learner-1", "memories")):
    print(" •", item.value["fact"])

## Lesson 5 — Semantic search over memories

With many memories you only want the **relevant** ones. Give the store an **embedding index**, then `search(ns, query="...")` returns the closest memories by meaning.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
smart_store = InMemoryStore(index={"embed": embeddings, "dims": 1536, "fields": ["fact"]})

ns = ("users", "u1", "memories")
for i, fact in enumerate([
    "Allergic to peanuts",
    "Works as a data scientist at a bank",
    "Has a pet dog named Bruno",
    "Prefers South Indian breakfast",
    "Is learning LangGraph in the evenings",
]):
    smart_store.put(ns, f"m{i}", {"fact": fact})

for q in ["what should I cook for them?", "tell me about their job"]:
    hits = smart_store.search(ns, query=q, limit=2)
    print(f"🔎 {q}")
    for h in hits:
        print(f"   {h.score:.2f}  {h.value['fact']}")

💡 For production use `PostgresStore` (package `langgraph-checkpoint-postgres`) — same API.

## Lesson 6 — Practice

**Exercise 1.** Write `save_pref(store, user_id, key, value)` and `get_prefs(store, user_id)` helpers using the namespace `("prefs", user_id)`. Save `theme=dark` and `lang=Hindi`, then read them back as a dict.

**Exercise 2.** Add a `language` to the `Context` when calling `memory_bot` and make the `chat` node reply in that language.

In [ ]:
# ✅ Solution 1
def save_pref(store, user_id, key, value):
    store.put(("prefs", user_id), key, {"value": value})

def get_prefs(store, user_id) -> dict:
    return {i.key: i.value["value"] for i in store.search(("prefs", user_id))}

s = InMemoryStore()
save_pref(s, "u9", "theme", "dark")
save_pref(s, "u9", "lang", "Hindi")
print(get_prefs(s, "u9"))
assert get_prefs(s, "u9") == {"theme": "dark", "lang": "Hindi"}

In [ ]:
# ✅ Solution 2
def chat_in_language(state: MessagesState, runtime: Runtime[Context]) -> dict:
    facts = [i.value["fact"] for i in runtime.store.search(("users", runtime.context.user_id, "memories"))]
    system = (f"Known facts about the user: {facts or 'none'}. "
              f"Always reply in {runtime.context.language}. Keep it short.")
    return {"messages": [llm.invoke([SystemMessage(content=system)] + state["messages"])]}

g = StateGraph(MessagesState, context_schema=Context)
g.add_node("chat", chat_in_language)
g.add_edge(START, "chat")
g.add_edge("chat", END)
lang_bot = g.compile(store=memory_store)
out = lang_bot.invoke({"messages": [("user", "Suggest a drink for me.")]},
                      context=Context(user_id="learner-1", language="Marathi"))
print(out["messages"][-1].content)
print("✅ All exercises solved!")

## 🎯 Day 14 Recap

* **Store** = long-term memory across threads: `put / get / search / delete` with namespaces.
* **Runtime context** = per-run settings (`context_schema=...`, `invoke(..., context=...)`, `runtime.context`).
* `compile(store=...)` → nodes use `runtime.store`.
* Pattern: **read** memories before answering, **extract & write** new facts after.
* Add an embedding `index` for semantic memory search.

**Tomorrow (Day 15):** production skills — retries, caching, error handling, the Functional API, testing and deployment.